In [26]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

import numpy as np

from utils.config import device

In [27]:
def get_races(path):
    races = np.load(path)
    result = []
    for race_id in races:
        result.append(torch.from_numpy(races[race_id]).to(device).double())

    return result


def get_reciprocal_ranks(path):
    race_ys = np.load(path)
    result = []
    for race_id in race_ys:
        race_y = torch.from_numpy(race_ys[race_id]).to(device).double()
        ranking = race_y[:, 0]
        number_of_horses = race_y[:, 4]

        result.append((number_of_horses - 1) / (ranking - 1))

    return result


races = get_races("../final_loaded_data/location_ST_1600/weighed/train/data_x.npz")
reciprocal_ranks = get_reciprocal_ranks("../final_loaded_data/location_ST_1600/weighed/train/data_y.npz")
assert len(races) == len(reciprocal_ranks)

len(races)

674

In [39]:
def pad_races(races, targets, device):
    padded_inputs = pad_sequence(races, batch_first=True).to(device)       # [B, H, 64]
    padded_targets = pad_sequence(targets, batch_first=True).to(device)    # [B, H]
    mask = torch.zeros(padded_targets.shape, dtype=torch.bool, device=device)
    for i, r in enumerate(targets):
        mask[i, :r.shape[0]] = 1
    return padded_inputs, padded_targets, mask

inputs, targets, mask = pad_races(races, reciprocal_ranks, device)

In [42]:
class ListwiseRanker(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x, mask):
        logits = self.model(x).squeeze(-1)               # [B, H]
        logits = logits.masked_fill(~mask, float('-inf'))  # Mask padded horses
        return logits

model = ListwiseRanker(64).to(device).double()

In [44]:
def masked_mse_loss(preds, targets, mask):
    loss = (preds - targets) ** 2
    loss = loss * mask.float()
    return loss.sum() / mask.sum()

logits = model(inputs, mask)             # [B, H]
loss = masked_mse_loss(logits, targets, mask)
loss

tensor(nan, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)